# Package import

In [82]:
import os
import yaml
import requests
from requests.auth import HTTPBasicAuth
from bs4 import BeautifulSoup
import pandas as pd 

# Apollo Scraper

In [13]:
with open('config.yaml','r', encoding="utf-8") as f:
    config=yaml.safe_load(f)
username=config["credentials"]["user"]
password=config["credentials"]["password"]
credentials=HTTPBasicAuth(username, password)

In [49]:
url = "https://planzajec.uek.krakow.pl/index.php?typ=G&id=252681&okres=2"
response = requests.get(url,auth = credentials)
response.encoding = 'utf-8'
print(response.status_code)

200


In [50]:
page_dom=BeautifulSoup(response.text, "html.parser")

In [85]:
group=page_dom.select_one("div.grupa").get_text(strip=True)
print(group)
# lectures=page_dom.find_all("th")
# print(lectures)

ZICSS1-1211


-----

In [73]:
headers=page_dom.find_all("th")
headers_points=[x.text for x in headers]

rows=page_dom.find_all("tr")[1:]
rows_points=[]
for row in rows:
    elements=row.find_all("td")
    if len(elements)>2:
        rows_points.append([x.text  for x in elements])
print(headers_points)
print(rows_points)
dp=pd.DataFrame(rows_points,columns=headers_points)
print(dp)
dp.to_csv("aa.csv", index=False, encoding='utf-8')

['Termin', 'Dzień, godzina', 'Przedmiot', 'Typ', 'Nauczyciel', 'Sala']
[['2026-02-23', 'Pn 11:30 - 13:00 (2g.)', 'Foreign Language I', 'lektorat', '', 'Wybierz swoją grupę językową'], ['2026-02-23', 'Pn 13:15 - 15:45 (3g.)', 'Computer Programming 2', 'ćwiczenia', 'dr Katarzyna Wójcik  ', 'Paw.A 013 lab. Win10, Office21'], ['2026-02-23', 'Pn 18:30 - 20:00 (2g.)', '', 'lektorat', '', 'Wybierz swoją grupę językową'], ['2026-02-24', 'Wt 09:45 - 11:15 (2g.)', 'Probability and Statistics', 'wykład', 'prof. dr hab. Andrzej Sokołowski  ', 'Paw.F 008'], ['2026-02-24', 'Wt 11:30 - 13:00 (2g.)', 'Foreign Language I', 'lektorat', '', 'Wybierz swoją grupę językową'], ['2026-02-24', 'Wt 13:15 - 14:45 (2g.)', 'Probability and Statistics', 'ćwiczenia', 'prof. dr hab. Andrzej Sokołowski  ', 'Paw.A 07 lab. Win7, Office21'], ['2026-02-24', 'Wt 15:00 - 16:30 (2g.)', 'Discrete mathematics', 'wykład', 'dr Grzegorz Kosiorowski  ', 'Rakowicka 16 sala 11'], ['2026-02-24', 'Wt 18:30 - 20:00 (2g.)', 'Business La

---

In [83]:
classes_tag=page_dom.select_one("table")
with open("temp.html", 'w', encoding="utf-8") as f:
    f.write(classes_tag.prettify())
classes=pd.read_html("temp.html", encoding="UTF-8")[0]
os.remove("temp.html")
print(classes)

         Termin          Dzień, godzina  \
0    2026-02-23  Pn 11:30 - 13:00 (2g.)   
1    2026-02-23  Pn 13:15 - 15:45 (3g.)   
2    2026-02-23  Pn 13:15 - 15:45 (3g.)   
3    2026-02-23  Pn 18:30 - 20:00 (2g.)   
4    2026-02-24  Wt 09:45 - 11:15 (2g.)   
..          ...                     ...   
293  2026-06-11  Cz 16:45 - 17:30 (1g.)   
294  2026-06-11  Cz 18:30 - 20:00 (2g.)   
295  2026-06-11  Cz 18:30 - 20:00 (2g.)   
296  2026-06-17  Śr 10:30 - 13:00 (3g.)   
297  2026-09-03  Cz 10:30 - 13:00 (3g.)   

                                   Przedmiot        Typ  \
0                         Foreign Language I   lektorat   
1                     Computer Programming 2  ćwiczenia   
2                                    CS gr 1    CS gr 1   
3                                        NaN   lektorat   
4                 Probability and Statistics     wykład   
..                                       ...        ...   
293                                  CS gr 1    CS gr 1   
294  Operat

In [84]:
if not os.path.exists("schedules"):
    os.mkdir("schedules")


In [86]:
classes.to_csv(f"schedules/{group}.csv")